<a href="https://colab.research.google.com/github/ChristianGoldbach/Gemini-LangGraph-for-Dummies/blob/main/Gu%C3%ADa_pr%C3%A1ctica_para_multiagentes_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain
!pip install langgraph
!pip install langchain_google_genai

In [ ]:
# Importaciones estándar
import os

# Importaciones externas
from langchain_core.messages import (BaseMessage, SystemMessage, HumanMessage, AIMessage, ToolMessage)
from langchain_google_genai import ChatGoogleGenerativeAI

#### Credenciales y Modelo de Lenguaje

In [ ]:
os.environ["GOOGLE_API_KEY"] = "your_actual_api_key_here"

llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    temperature=0.3,
    # max_tokens=1000,
)

In [ ]:
llm.invoke("hola").content

# Grafo

## state.py

In [ ]:
from typing import Annotated, Any, Sequence, TypedDict
from langgraph.graph.message import add_messages

In [ ]:
class AgentState(TypedDict):
  messages: Annotated[Sequence[BaseMessage], add_messages]
  final_result: str

## models.py

In [ ]:
from typing import List
from pydantic import BaseModel
from langchain_core.output_parsers import JsonOutputParser

## prompts.py

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

pep8_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """You are a helpful AI assistant that acts as a Python code formatter.
Your task is to take the provided Python code and reformat it according to PEP 8 style guidelines.
This includes but is not limited to:
- Consistent indentation (typically 4 spaces).
- Lines limited to a maximum length (e.g., 79 characters for code, 72 for comments and docstrings).
- Proper spacing around operators and after commas.
- Blank lines used appropriately to separate logical sections of code.
- Docstrings formatted according to conventions (e.g., Google style).
- Consistent naming conventions (e.g., lower_case_with_underscores for variables and functions, PascalCase for classes).
- Removing any unnecessary whitespace or inconsistent formatting.

You should only return the reformatted Python code. Do not add any explanations or surrounding text unless the input is not valid Python code. If the input is not valid Python code, inform the user that you can only format Python code.
""",
        ),
        ("user", "{input_code}"),
    ]
)

## nodes.py

In [ ]:
def pep8_assistant(state: AgentState):
  messages = state["messages"]

  chain = pep8_prompt | llm

  response = chain.invoke({"input_code": messages[-1].content})

  return {
      "final_result": response.content
  }

## graph.py

In [ ]:
from langgraph.graph import START, END, StateGraph

builder = StateGraph(AgentState)

# Definir los nodos y las conexiones del grafo
builder.add_edge(START, "PEP 8 Assistant")
builder.add_node("PEP 8 Assistant", pep8_assistant)
builder.add_edge("PEP 8 Assistant", END)

code_refiner = builder.compile()

In [ ]:
# from IPython.display import Image, display

# # Method 1: Default Mermaid API
# display(Image(code_refiner.get_graph().draw_mermaid_png()))

## Ejecución del grafo

In [ ]:
from langchain_core.messages import HumanMessage

In [ ]:
# @title
response = code_refiner.invoke(
    {
        "messages": [
            HumanMessage(
                content = """def add_numbers(a, b):
                  return a + b"""
            )
        ]
    }
)

In [ ]:
response['final_result']

In [ ]:
def add_numbers(a, b):
    """Adds two numbers together.

    Args:
        a: The first number.
        b: The second number.

    Returns:
        The sum of a and b.
    """
    return a + b